## PCA Analysis

In [39]:
%matplotlib inline
import yfinance as yf
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import ttest_ind, f_oneway
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Import Ensemble models
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor 
from sklearn.metrics import classification_report, accuracy_score
import cvxpy as cp
import re
import os
import joblib
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
from matplotlib import cm
from sklearn.metrics import precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from statsmodels.tsa.stattools import ccf
from grid_chart_util import create_stock_price_grid
from spark_init import initialize_spark
from config_loader import load_config
from pandas_util import convert_to_yfinance_format
from finance_util import calculate_technical_indicators
from datetime import datetime, timedelta, date

In [40]:
# Get date threshold (last 3 weeks)
# date_threshold = datetime.today() - timedelta(weeks=26)
# Get today's date and replace the month and day to January 1st
# date_threshold = datetime.today().replace(month=1, day=1)
# Get today's date and calculate December 1 of the previous year
date_threshold = datetime(datetime.today().year - 1, 12, 1)
print(f"date_threshold : {date_threshold}")
# Create directory if it doesn't exist
save_dir = 'model/data'
os.makedirs(save_dir, exist_ok=True)
filename = "model/data/combined_output.csv"
print(f"Filename to read : {filename}")
today = date.today()
end_date = today.strftime("%Y-%m-%d")
print(f"end_date : {end_date}")

date_threshold : 2024-12-01 00:00:00
Filename to read : model/data/combined_output.csv
end_date : 2025-02-23


## I. PCA Analysis

### 1. Fetch data for the FAANG stocks plus additional promising stocks

In [41]:
# Read and clean data
historical_data = pd.read_csv(filename, header=[0, 1], index_col=0, parse_dates=True)
historical_data = historical_data.drop_duplicates()
historical_data.columns.names = ['Ticker', 'Price']  # Fix: Ticker first, then Price

# Verify structure
print(historical_data.columns.levels)
print(historical_data.columns.names)
# Extract unique tickers from the 'Ticker' level
tickers = historical_data.columns.get_level_values('Ticker').unique().tolist()
print("Tickers to process:", tickers)

[['AAPL', 'ACB', 'AMZN', 'BB', 'BGNE', 'CLOV', 'CLS', 'CRNC', 'DDD', 'DOCS', 'EDIT', 'HOOD', 'LEDS', 'MNDY', 'NBIS', 'NET', 'NVDA', 'NVRO', 'OPFI', 'PLBY', 'PLTR', 'ROOT', 'RXRX'], ['Close', 'High', 'Low', 'Open', 'Volume']]
['Ticker', 'Price']
Tickers to process: ['ROOT', 'NVRO', 'HOOD', 'AAPL', 'CRNC', 'DOCS', 'OPFI', 'LEDS', 'NET', 'BGNE', 'EDIT', 'PLTR', 'MNDY', 'CLS', 'DDD', 'PLBY', 'RXRX', 'AMZN', 'NBIS', 'ACB', 'NVDA', 'BB', 'CLOV']


## Data Preparation as Yahoo

### 2. Feature Engineering

In [42]:
tickers = ['AMZN', 'NVDA', 'AAPL']  # Adjust to your full list
future_features = pd.DataFrame()
for ticker in tickers:
    stock_data = historical_data.xs(ticker, level='Ticker', axis=1).copy()
    stock_data = calculate_technical_indicators(stock_data)
    selected_cols = ['Return', '20d_MA', '50d_MA', '20d_Volatility', 'Upper_BB', 'Lower_BB', 'RSI', 'MACD', 'Signal_Line', 'OBV', 'ATR', 'Williams_%R']
    stock_data = stock_data[selected_cols]
    stock_data.columns = [f'{ticker}_{col}' for col in stock_data.columns]
    if future_features.shape[0] == 0 and future_features.shape[1] == 0:
        future_features = stock_data
    else:
        future_features = future_features.join(stock_data, how='outer')

future_features = future_features.fillna(0)
#print(type(future_features))
#print(future_features.head())

### 3. Combine Features for Each Ticker

### 4. Feature Selection

#### 4.1 Correlation Analysis

In [43]:
# Assuming future_features and tickers are defined
print("Type of future_features:", type(future_features))
print("Columns in future_features:", future_features.columns.tolist())

Type of future_features: <class 'pandas.core.frame.DataFrame'>
Columns in future_features: ['AMZN_Return', 'AMZN_20d_MA', 'AMZN_50d_MA', 'AMZN_20d_Volatility', 'AMZN_Upper_BB', 'AMZN_Lower_BB', 'AMZN_RSI', 'AMZN_MACD', 'AMZN_Signal_Line', 'AMZN_OBV', 'AMZN_ATR', 'AMZN_Williams_%R', 'NVDA_Return', 'NVDA_20d_MA', 'NVDA_50d_MA', 'NVDA_20d_Volatility', 'NVDA_Upper_BB', 'NVDA_Lower_BB', 'NVDA_RSI', 'NVDA_MACD', 'NVDA_Signal_Line', 'NVDA_OBV', 'NVDA_ATR', 'NVDA_Williams_%R', 'AAPL_Return', 'AAPL_20d_MA', 'AAPL_50d_MA', 'AAPL_20d_Volatility', 'AAPL_Upper_BB', 'AAPL_Lower_BB', 'AAPL_RSI', 'AAPL_MACD', 'AAPL_Signal_Line', 'AAPL_OBV', 'AAPL_ATR', 'AAPL_Williams_%R']


In [44]:
# Your original code
target_returns = future_features[[f'{ticker}_Return' for ticker in tickers]].mean(axis=1)
correlation_matrix = future_features.corrwith(target_returns)
correlation_threshold = 0.1
selected_features = correlation_matrix[abs(correlation_matrix) > correlation_threshold].index
future_features = future_features[selected_features]

print("Correlation Matrix:")
#print(correlation_matrix)
print("\nSelected Features (abs(corr) > 0.1):")
#print(selected_features)
#print("Updated future_features shape:", future_features.shape)

Correlation Matrix:

Selected Features (abs(corr) > 0.1):


### 5. Train-Test Splitting and Normalization

In [45]:
def prepare_and_test_features(df, tickers):
    stats_df = df.copy()
    available_returns = [col for col in df.columns if col.endswith('_Return')]
    if not available_returns:
        print("Warning: No return columns remain after correlation filtering. Using Target_Return as fallback.")
        stats_df['Target_Return'] = target_returns
    else:
        stats_df['Target_Return'] = stats_df[available_returns].mean(axis=1)
    
    t_stats = {}
    p_values = {}
    numeric_cols = [col for col in stats_df.columns if col != 'Target_Return']
    for col in numeric_cols:
        t_stat, p_val = stats.ttest_rel(stats_df[col], stats_df['Target_Return'])
        t_stats[col] = t_stat
        p_values[col] = p_val
    significant_features = [col for col, p in p_values.items() if p <= 0.05]
    stats_df = stats_df[significant_features + ['Target_Return']]
    
    print("\nT-test Results:")
    for col in t_stats:
        print(f"{col}: t-stat = {t_stats[col]:.4f}, p-value = {p_values[col]:.4f}")
    print("\nFeatures kept after t-test (p <= 0.05):", significant_features)
    
    anova_results = {}
    for col in significant_features:
        quartiles = pd.qcut(stats_df[col], q=4, labels=False, duplicates='drop')
        groups = [stats_df['Target_Return'][quartiles == i] for i in range(quartiles.max() + 1)]
        f_stat, p_val = stats.f_oneway(*groups)
        anova_results[col] = {'f_stat': f_stat, 'pvalue': p_val}
    final_features = [col for col, res in anova_results.items() if res['pvalue'] <= 0.05]
    stats_df = stats_df[final_features + ['Target_Return']]
    
    print("\nANOVA Results:")
    for col in anova_results:
        print(f"{col}: F-stat = {anova_results[col]['f_stat']:.4f}, p-value = {anova_results[col]['pvalue']:.4f}")
    print("\nFinal features after ANOVA (p <= 0.05):", final_features)
    return stats_df

stats_df = prepare_and_test_features(future_features, tickers)


T-test Results:
AMZN_Return: t-stat = 0.1036, p-value = 0.9178
AMZN_OBV: t-stat = 15.6462, p-value = 0.0000
AMZN_Williams_%R: t-stat = -8.2044, p-value = 0.0000
NVDA_Return: t-stat = -0.0959, p-value = 0.9240
NVDA_Signal_Line: t-stat = -2.5822, p-value = 0.0126
NVDA_OBV: t-stat = -11.5904, p-value = 0.0000
NVDA_Williams_%R: t-stat = -8.8553, p-value = 0.0000
AAPL_Return: t-stat = 0.0543, p-value = 0.9569
AAPL_Williams_%R: t-stat = -8.1750, p-value = 0.0000

Features kept after t-test (p <= 0.05): ['AMZN_OBV', 'AMZN_Williams_%R', 'NVDA_Signal_Line', 'NVDA_OBV', 'NVDA_Williams_%R', 'AAPL_Williams_%R']

ANOVA Results:
AMZN_OBV: F-stat = 0.6856, p-value = 0.5650
AMZN_Williams_%R: F-stat = 2.8840, p-value = 0.0446
NVDA_Signal_Line: F-stat = 2.7147, p-value = 0.0544
NVDA_OBV: F-stat = 0.7407, p-value = 0.5327
NVDA_Williams_%R: F-stat = 0.4854, p-value = 0.6939
AAPL_Williams_%R: F-stat = 0.0686, p-value = 0.9764

Final features after ANOVA (p <= 0.05): ['AMZN_Williams_%R']


### 6. Perform PCA Analysis

### 7. Interpret Principal Components

In [46]:
def create_sequences(data, seq_length, forecast_horizon):
    X, y = [], []
    for i in range(len(data) - seq_length - forecast_horizon + 1):
        X.append(data.drop(columns=['Target_Return']).iloc[i:(i + seq_length)].values)
        y.append(data['Target_Return'].iloc[(i + seq_length):(i + seq_length + forecast_horizon)].values)
    return np.array(X), np.array(y)

seq_length = 20
forecast_horizon = 14
X, y = create_sequences(stats_df, seq_length, forecast_horizon)

In [47]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [48]:
# Flatten X for ensemble models
n_samples_train, seq_len, n_features = X_train.shape
n_samples_test = X_test.shape[0]
X_train_flat = X_train.reshape(n_samples_train, seq_len * n_features)
X_test_flat = X_test.reshape(n_samples_test, seq_len * n_features)

### 8. Get all Feature Loadings

In [49]:
# Preprocessing
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)

### 9. Identify the Top Features for Each Principal Component

### 10. Plot the Loadings for the First Two Principal Components

### 11. Plot the Top Features for Each Principal Component

### Step 8: Identify Interesting Principal Components

In [50]:
pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

## II Stock Selection - first reduction using ensemble models

In [51]:
def train_evaluate_select_model(X_train, X_val, y_train, y_val):
    models = [
        ('Random Forest', RandomForestRegressor(random_state=42)),
        ('Gradient Boosting', GradientBoostingRegressor(random_state=42)),
        ('AdaBoost', AdaBoostRegressor(random_state=42)),
        ('XGBoost', XGBRegressor(random_state=42)),
        ('Extra Trees', ExtraTreesRegressor(random_state=42))
    ]
    best_model = None
    best_model_name = None
    best_mse = float('inf')
    for name, model in models:
        model.fit(X_train, y_train)
        val_predictions = model.predict(X_val)
        mse = np.mean((val_predictions - y_val) ** 2)
        print(f"{name} Validation MSE: {mse:.4f}")
        if mse < best_mse:
            best_mse = mse
            best_model = model
            best_model_name = name
    return best_model, best_model_name

### 1. Model Selection

In [52]:
# Train and save a model for each day
models = []
for day in range(forecast_horizon):
    print(f"\nTraining model for day {day + 1}")
    model, model_name = train_evaluate_select_model(X_train_pca, X_test_pca, y_train[:, day], y_test[:, day])
    models.append((model, model_name))
    # Save each model in the specified folder
    joblib.dump(model, os.path.join(save_dir, f'model_day_{day + 1}.pkl'))
    #print(f"Best model for day {day + 1}: {model_name} saved as '{save_dir}/model_day_{day + 1}.pkl'")

# Save preprocessing objects in the specified folder
joblib.dump(scaler, os.path.join(save_dir, 'scaler.pkl'))
joblib.dump(pca, os.path.join(save_dir, 'pca.pkl'))
joblib.dump(stats_df.drop(columns=['Target_Return']).columns, os.path.join(save_dir, 'feature_names.pkl'))

# Get latest data for prediction (example output)
latest_features = stats_df.drop(columns=['Target_Return']).iloc[-seq_length:].values.flatten().reshape(1, -1)
latest_scaled = scaler.transform(latest_features)
latest_pca = pca.transform(latest_scaled)


Training model for day 1
Random Forest Validation MSE: 3.2222
Gradient Boosting Validation MSE: 3.3424
AdaBoost Validation MSE: 3.1578
XGBoost Validation MSE: 3.7640
Extra Trees Validation MSE: 3.5368

Training model for day 2
Random Forest Validation MSE: 9.6895
Gradient Boosting Validation MSE: 13.5276
AdaBoost Validation MSE: 7.3246
XGBoost Validation MSE: 14.8221
Extra Trees Validation MSE: 11.2778

Training model for day 3
Random Forest Validation MSE: 11.8570
Gradient Boosting Validation MSE: 16.6721
AdaBoost Validation MSE: 13.5232
XGBoost Validation MSE: 19.2531
Extra Trees Validation MSE: 13.7826

Training model for day 4
Random Forest Validation MSE: 10.9475
Gradient Boosting Validation MSE: 14.7971
AdaBoost Validation MSE: 12.1752
XGBoost Validation MSE: 12.7680
Extra Trees Validation MSE: 10.3361

Training model for day 5
Random Forest Validation MSE: 8.1443
Gradient Boosting Validation MSE: 13.6259
AdaBoost Validation MSE: 8.4930
XGBoost Validation MSE: 15.3167
Extra Tree

In [53]:
# Save preprocessing objects in the specified folder
joblib.dump(scaler, os.path.join(save_dir, 'scaler.pkl'))
joblib.dump(pca, os.path.join(save_dir, 'pca.pkl'))
joblib.dump(stats_df.drop(columns=['Target_Return']).columns, os.path.join(save_dir, 'feature_names.pkl'))

# Get latest data for prediction (example output)
latest_features = stats_df.drop(columns=['Target_Return']).iloc[-seq_length:].values.flatten().reshape(1, -1)
latest_scaled = scaler.transform(latest_features)
latest_pca = pca.transform(latest_scaled)

In [54]:
# Predict and save example forecast
predicted_returns = [model.predict(latest_pca)[0] for model, _ in models]
available_returns = [col for col in future_features.columns if col.endswith('_Return')]
if available_returns:
    volatility = future_features[available_returns].std().mean()
else:
    volatility = future_features[[f'{ticker}_20d_Volatility' for ticker in tickers if f'{ticker}_20d_Volatility' in future_features.columns]].mean().mean()

#current_prices = {ticker: historical_data['Close'][ticker].iloc[-1] for ticker in tickers}
# Get current (last) prices
current_prices = {ticker: historical_data[(ticker, 'Close')].iloc[-1] for ticker in tickers}
print("Current prices:", current_prices)
forecast_dates = pd.date_range(start=end_date, periods=15)[1:]

Current prices: {'AMZN': 216.5800018310547, 'NVDA': 134.42999267578125, 'AAPL': 245.5500030517578}


### 2. Stock selection with high growth potential

In [55]:
tickers_for_prediction = ['NVDA']
print("Tickers to process:", tickers_for_prediction)
for ticker in tickers_for_prediction:
    current_price = current_prices[ticker]
    forecast_prices = [current_price]
    forecast_lows = [current_price]
    forecast_highs = [current_price]
    for i in range(14):
        daily_return = predicted_returns[i]
        next_price = forecast_prices[-1] * (1 + daily_return / 100)
        daily_vol = volatility * forecast_prices[-1] / 100
        forecast_prices.append(next_price)
        forecast_lows.append(next_price - daily_vol)
        forecast_highs.append(next_price + daily_vol)
    
    forecast_df = pd.DataFrame({
        'Date': forecast_dates,
        'Predicted_Price': forecast_prices[1:],
        'Low': forecast_lows[1:],
        'High': forecast_highs[1:]
    })
    print(f"\n{ticker} 14-Day Price Forecast:")
    print(forecast_df)

Tickers to process: ['NVDA']

NVDA 14-Day Price Forecast:
         Date  Predicted_Price         Low        High
0  2025-02-24       133.515348  130.437252  136.593443
1  2025-02-25       130.412684  127.355531  133.469836
2  2025-02-26       130.810486  127.824377  133.796595
3  2025-02-27       128.526658  125.531440  131.521876
4  2025-02-28       129.177223  126.234299  132.120147
5  2025-03-01       129.471033  126.513213  132.428854
6  2025-03-02       130.875189  127.910641  133.839737
7  2025-03-03       129.960913  126.964213  132.957612
8  2025-03-04       129.689734  126.713969  132.665499
9  2025-03-05       129.938053  126.968497  132.907609
10 2025-03-06       129.905936  126.930695  132.881178
11 2025-03-07       128.808606  125.834100  131.783112
12 2025-03-08       126.531912  123.582532  129.481292
13 2025-03-09       125.402635  122.505385  128.299885
